In [3]:
import pandas as pd
import numpy as np
import re


In [4]:
# ===============================
# 1. Carregar os dados
# ===============================
df = pd.read_csv("fifa21_raw_data.csv", low_memory=False)


In [5]:
print("Dimensões do dataset:")
print(df.shape)

Dimensões do dataset:
(18979, 77)


In [6]:
# ===============================
# 2. Informações gerais
# ===============================
print("\nInformações:")
print(df.info())

print("\nValores nulos:")
print(df.isnull().sum())



Informações:
<class 'pandas.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   photoUrl          18979 non-null  str  
 1   LongName          18979 non-null  str  
 2   playerUrl         18979 non-null  str  
 3   Nationality       18979 non-null  str  
 4   Positions         18979 non-null  str  
 5   Name              18979 non-null  str  
 6   Age               18979 non-null  int64
 7   ↓OVA              18979 non-null  int64
 8   POT               18979 non-null  int64
 9   Team & Contract   18979 non-null  str  
 10  ID                18979 non-null  int64
 11  Height            18979 non-null  str  
 12  Weight            18979 non-null  str  
 13  foot              18979 non-null  str  
 14  BOV               18979 non-null  int64
 15  BP                18979 non-null  str  
 16  Growth            18979 non-null  int64
 17  Joined            18979 non-

In [7]:
# ===============================
# 3. Remover duplicatas
# ===============================
df.drop_duplicates(inplace=True)


In [8]:
# ===============================
# 4. Remover espaços em branco
# ===============================
df.columns = df.columns.str.strip()

In [9]:
# ===============================
# 5. Converter idade
# ===============================
if "Age" in df.columns:
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")


In [10]:
# ===============================
# 6. Função para converter dinheiro
# ===============================
def converter_dinheiro(valor):

    if pd.isna(valor):
        return np.nan

    valor = str(valor).replace("€", "")

    if "M" in valor:
        return float(valor.replace("M", "")) * 1_000_000

    elif "K" in valor:
        return float(valor.replace("K", "")) * 1_000

    else:
        try:
            return float(valor)
        except:
            return np.nan

# Converter Value
if "Value" in df.columns:
    df["Value"] = df["Value"].apply(converter_dinheiro)

# Converter Wage
if "Wage" in df.columns:
    df["Wage"] = df["Wage"].apply(converter_dinheiro)

# Converter Release Clause
if "Release Clause" in df.columns:
    df["Release Clause"] = df["Release Clause"].apply(converter_dinheiro)


In [11]:
# ===============================
# 7. Converter altura
# Ex.: 5'11" -> centímetros
# ===============================
def altura_cm(texto):

    if pd.isna(texto):
        return np.nan

    try:
        partes = texto.replace('"', "").split("'")
        pes = int(partes[0])
        polegadas = int(partes[1])

        return round((pes*30.48)+(polegadas*2.54),1)

    except:
        return np.nan

if "Height" in df.columns:
    df["Height(cm)"] = df["Height"].apply(altura_cm)


In [13]:
# ===============================
# 8. Converter peso
# ===============================
def peso_kg(texto):

    if pd.isna(texto):
        return np.nan

    texto = texto.replace("lbs","")

    try:
        return round(float(texto)*0.453592,1)
    except:
        return np.nan

if "Weight" in df.columns:
    df["Weight(kg)"] = df["Weight"].apply(peso_kg)

In [14]:
# ===============================
# 9. Corrigir coluna Hits
# ===============================
if "Hits" in df.columns:

    df["Hits"] = (
        df["Hits"]
        .astype(str)
        .str.replace("\n","",regex=False)
        .str.strip()
    )

    def converter_hits(valor):

        if pd.isna(valor):
            return np.nan

        valor = str(valor)

        if "K" in valor:
            return float(valor.replace("K",""))*1000

        try:
            return float(valor)
        except:
            return np.nan

    df["Hits"] = df["Hits"].apply(converter_hits)


In [16]:
# ===============================
# 10. Converter datas
# ===============================
if "Joined" in df.columns:
    df["Joined"] = pd.to_datetime(df["Joined"], errors="coerce")

if "Loan Date End" in df.columns:
    df["Loan Date End"] = pd.to_datetime(df["Loan Date End"], errors="coerce")


In [18]:
# ===============================
# 11. Completar valores ausentes
# ===============================
numericas = df.select_dtypes(include=np.number).columns

for coluna in numericas:
    df[coluna] = df[coluna].fillna(df[coluna].median())

categoricas = df.select_dtypes(include=["object", "string"]).columns

for coluna in categoricas:
    df[coluna] = df[coluna].fillna("Desconhecido")

In [19]:
# ===============================
# 12. Criar coluna IMC
# ===============================
if "Height(cm)" in df.columns and "Weight(kg)" in df.columns:

    altura_m = df["Height(cm)"] / 100

    df["BMI"] = df["Weight(kg)"] / (altura_m ** 2)


In [20]:
# ===============================
# 13. Resumo final
# ===============================
print("\nValores nulos restantes:")
print(df.isnull().sum())

print("\nTipos das colunas:")
print(df.dtypes)

print("\nPrimeiras linhas:")
print(df.head())


Valores nulos restantes:
photoUrl       0
LongName       0
playerUrl      0
Nationality    0
Positions      0
              ..
PHY            0
Hits           0
Height(cm)     0
Weight(kg)     0
BMI            0
Length: 80, dtype: int64

Tipos das colunas:
photoUrl           str
LongName           str
playerUrl          str
Nationality        str
Positions          str
                ...   
PHY              int64
Hits           float64
Height(cm)     float64
Weight(kg)     float64
BMI            float64
Length: 80, dtype: object

Primeiras linhas:
                                           photoUrl  \
0  https://cdn.sofifa.com/players/158/023/21_60.png   
1  https://cdn.sofifa.com/players/020/801/21_60.png   
2  https://cdn.sofifa.com/players/200/389/21_60.png   
3  https://cdn.sofifa.com/players/192/985/21_60.png   
4  https://cdn.sofifa.com/players/190/871/21_60.png   

                       LongName  \
0                  Lionel Messi   
1  C. Ronaldo dos Santos Aveiro   
2       

In [22]:
# ===============================
# 14. Salvar dataset limpo
# ===============================
df.to_csv("fifa21_limpo.csv", index=False)

print("\nArquivo salvo como:")
print("fifa21_limpo.csv")


Arquivo salvo como:
fifa21_limpo.csv
